# Week 35, Day 4: Adapter

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w035d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def count_words(source):
    return len(source.read().split())

class LinesSource:
    def __init__(self, lines):
        self.lines = lines

    def read(self):
        return "\n".join(self.lines)

import io
print(count_words(io.StringIO("Ẹ káàbọ̀ to the course")))
print(count_words(LinesSource(["three words here", "and two"])))

In [ ]:
from typing import Protocol

class Sensor(Protocol):
    def celsius(self) -> float: ...

class DigitalSensor:                        # already fits
    def __init__(self, c):
        self.c = c
    def celsius(self):
        return self.c

class OldThermometer:                       # can't change: reports tenths of a degree Fahrenheit
    def __init__(self, tenths_f):
        self.tenths_f = tenths_f
    def read_f_tenths(self):
        return self.tenths_f

class ThermometerAdapter:                   # makes an OldThermometer look like a Sensor
    def __init__(self, thermometer):
        self.thermometer = thermometer

    def celsius(self):
        f = self.thermometer.read_f_tenths() / 10
        return round((f - 32) * 5 / 9, 1)

def average(sensors: list[Sensor]) -> float:
    return round(sum(s.celsius() for s in sensors) / len(sensors), 1)

print(average([DigitalSensor(20.5), ThermometerAdapter(OldThermometer(707))]))

In [ ]:
from functools import partial

def send(channel, to, text):
    return f"[{channel}] to {to}: {text}"

def notify_all(handlers, message):
    return [h(message) for h in handlers]

handlers = [
    lambda text: send("sms", "07700 900123", text),         # a lambda as an adapter
    partial(send, "email", "ada@example.org"),               # partial fills in the first arguments
    partial(send, "chat", "ops-team"),
]
for line in notify_all(handlers, "Server room is 31°C"):
    print(line)

In [ ]:
from functools import partial

def send(channel, to, text):
    return f"[{channel}] to {to}: {text}"

p = partial(send, "email", to="ada@example.org")
print(p.func.__name__, p.args, p.keywords)
print(p(text="Your parcel has arrived"))

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Forecast:
    city: str
    high_c: float
    low_c: float
    rain_mm: float

def from_skycast(payload):
    return Forecast(payload["location"]["name"], payload["temp"]["max"], payload["temp"]["min"], payload.get("precip", 0.0))

def from_weathernest(payload):
    f_to_c = lambda f: round((f - 32) * 5 / 9, 1)
    hi, lo = payload["range_f"]
    return Forecast(payload["city"].title(), f_to_c(hi), f_to_c(lo), payload["rain_in"] * 25.4)

a = from_skycast({"location": {"name": "Lagos"}, "temp": {"max": 31.0, "min": 24.5}, "precip": 12.0})
b = from_weathernest({"city": "leeds", "range_f": [50, 41], "rain_in": 0.2})
for f in (a, b):
    print(f"{f.city}: {f.low_c} to {f.high_c}°C, {f.rain_mm:.1f} mm rain")

In [ ]:
class Stock:
    def __init__(self, levels):
        self.levels = dict(levels)
    def take(self, item, qty):
        if self.levels.get(item, 0) < qty:
            raise ValueError(f"not enough {item}")
        self.levels[item] -= qty
    def put_back(self, item, qty):
        self.levels[item] += qty

class Payments:
    def charge(self, customer, pence):
        if pence > 10000:
            raise ValueError("card declined")
        return f"PAY-{customer}-{pence}"

class Mailer:
    def __init__(self):
        self.sent = []
    def send(self, to, text):
        self.sent.append((to, text))

class Shop:                                  # the facade
    def __init__(self, stock, payments, mailer, prices):
        self.stock, self.payments, self.mailer, self.prices = stock, payments, mailer, prices

    def buy(self, customer, item, qty):
        self.stock.take(item, qty)
        try:
            ref = self.payments.charge(customer, self.prices[item] * qty)
        except ValueError:
            self.stock.put_back(item, qty)
            raise
        self.mailer.send(customer, f"Thanks! {qty} x {item}, ref {ref}")
        return ref

shop = Shop(Stock({"rice": 5}), Payments(), Mailer(), {"rice": 4500})
print(shop.buy("ada", "rice", 2))
try:
    shop.buy("bo", "rice", 3)
except ValueError as err:
    print("ValueError:", err, "| rice left:", shop.stock.levels["rice"])
print(shop.mailer.sent)

> **Key idea.** Adapter (Gang of Four): a small translator that makes something with the wrong interface fit the one your code expects, without changing either side. Prefer object adapters, which hold the adapted object and translate calls, names and units, over adapters that inherit. Adapt functions' signatures with lambdas or `functools.partial`, which fills in some arguments in advance. Adapt outside data with one converter function per source, at the boundary, into your own dataclasses. Facade puts one simple interface in front of several parts, coordinating them.

### Exercise: An adapter for an old sensor

`average_celsius(sensors)` works with anything that has a `celsius()` method. Write `KelvinAdapter`, which makes a `KelvinProbe` (which you can't change) fit: `celsius()` is the probe's reading minus 273.15, rounded to 1 place.

In [ ]:
class KelvinProbe:
    def __init__(self, kelvin):
        self.kelvin = kelvin

    def measure_k(self):
        return self.kelvin

class Digital:
    def __init__(self, c):
        self.c = c

    def celsius(self):
        return self.c

def average_celsius(sensors):
    return round(sum(s.celsius() for s in sensors) / len(sensors), 1)

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_adapter():\n    \"adapting\"\n    K, A, D = ns[\"KelvinProbe\"], ns[\"KelvinAdapter\"], ns[\"Digital\"]\n    assert A(K(293.15)).celsius() == 20.0 and A(K(273.15)).celsius() == 0.0\n    assert average_celsius([D(19.0), A(K(296.15))]) == 21.0\n    assert not issubclass(A, K), \"hold the probe; don't inherit from it\"\n    probe = K(300)\n    adapter = A(probe)\n    probe.kelvin = 310\n    assert adapter.celsius() == 36.9, \"read the probe each time\"")

### Exercise: Callbacks that fit

`Alarm.trigger(room)` calls each of its handlers with just the room name. Using `functools.partial`, write `make_handlers()`, which returns three handlers built from the given functions: one that texts `07700 900123`, one that emails `security@example.org`, and one that logs at the level `"ALERT"`. Each returns the text its function returns.

In [ ]:
from functools import partial

def text(number, room):
    return f"SMS {number}: alarm in {room}"

def email(address, room, subject="Alarm"):
    return f"EMAIL {address} [{subject}]: alarm in {room}"

def log(room, level="INFO"):
    return f"{level}: alarm in {room}"

class Alarm:
    def __init__(self, handlers):
        self.handlers = handlers

    def trigger(self, room):
        return [h(room) for h in self.handlers]

def make_handlers():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_partials():\n    \"partials\"\n    from functools import partial as P\n    hs = make_handlers()\n    assert all(isinstance(h, P) for h in hs), \"use functools.partial\"\n    assert ns[\"Alarm\"](hs).trigger(\"Kitchen\") == [\"SMS 07700 900123: alarm in Kitchen\", \"EMAIL security@example.org [Alarm]: alarm in Kitchen\", \"ALERT: alarm in Kitchen\"]")

### Exercise: Data from two services

Two weather services send forecasts in different shapes. Write data adapters into the `Forecast` dataclass:

- `from_skycast(payload)`: payloads like `{"location": {"name": "Lagos"}, "temp": {"max": 31.0, "min": 24.5}, "precip": 12.0}`; `"precip"` may be missing (meaning 0.0).
- `from_weathernest(payload)`: payloads like `{"city": "leeds", "range_f": [50, 41], "rain_in": 0.2}`: the city is in lower case (title-case it), temperatures are high then low in Fahrenheit (convert, rounding to 1 place), and rain is in inches (1 inch is 25.4 mm; round to 1 place).

Then write `wettest(forecasts)`, the city with the most rain (the first on a tie), or `None` for none. A payload missing a required key raises `ValueError` (not `KeyError`).

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Forecast:
    city: str
    high_c: float
    low_c: float
    rain_mm: float

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_adapters():\n    \"adapting data\"\n    F = ns[\"Forecast\"]\n    a = from_skycast({\"location\": {\"name\": \"Lagos\"}, \"temp\": {\"max\": 31.0, \"min\": 24.5}, \"precip\": 12.0})\n    b = from_weathernest({\"city\": \"leeds\", \"range_f\": [50, 41], \"rain_in\": 0.2})\n    c = from_skycast({\"location\": {\"name\": \"Kano\"}, \"temp\": {\"max\": 36.0, \"min\": 22.0}})\n    assert a == F(\"Lagos\", 31.0, 24.5, 12.0) and b == F(\"Leeds\", 10.0, 5.0, 5.1) and c.rain_mm == 0.0\n    assert wettest([b, a, c]) == \"Lagos\" and wettest([]) is None and wettest([c, F(\"Abuja\", 30, 20, 0.0)]) == \"Kano\"\n    for bad in [lambda: from_skycast({\"temp\": {\"max\": 1, \"min\": 0}}), lambda: from_weathernest({\"city\": \"x\", \"rain_in\": 0})]:\n        try:\n            bad()\n            assert False, \"a missing key should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: A facade for booking

Booking a fitness class needs three parts: `Classes` (spaces per class), `Wallet` (members' credits) and `Messages` (a log). Write a facade `Gym(classes, wallet, messages)` with `book(member, class_name)`, which:

1. takes a space (`Classes.take` raises `ValueError` if the class is full or unknown);
2. charges one credit (`Wallet.charge` raises `ValueError` if the member has none); if that fails, it gives the space back before letting the error through;
3. sends the member `"Booked yoga"` (with the class's name) and returns `True`.

In [ ]:
class Classes:
    def __init__(self, spaces):
        self.spaces = dict(spaces)
    def take(self, name):
        if self.spaces.get(name, 0) < 1:
            raise ValueError(f"{name} is full")
        self.spaces[name] -= 1
    def give_back(self, name):
        self.spaces[name] += 1

class Wallet:
    def __init__(self, credits):
        self.credits = dict(credits)
    def charge(self, member):
        if self.credits.get(member, 0) < 1:
            raise ValueError(f"{member} has no credits")
        self.credits[member] -= 1

class Messages:
    def __init__(self):
        self.sent = []
    def send(self, member, text):
        self.sent.append((member, text))

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gym():\n    \"the facade\"\n    C, W, M, G = ns[\"Classes\"], ns[\"Wallet\"], ns[\"Messages\"], ns[\"Gym\"]\n    g = G(C({\"yoga\": 1, \"spin\": 5}), W({\"ada\": 2, \"bo\": 0}), M())\n    assert g.book(\"ada\", \"yoga\") is True and g.messages.sent == [(\"ada\", \"Booked yoga\")]\n    for member, cls in [(\"ada\", \"yoga\"), (\"bo\", \"spin\"), (\"ada\", \"boxing\")]:\n        try:\n            g.book(member, cls)\n            assert False, f\"{member} booking {cls} should raise ValueError\"\n        except ValueError:\n            pass\n    assert g.classes.spaces == {\"yoga\": 0, \"spin\": 5}, \"a failed payment must give the space back\"\n    assert g.wallet.credits == {\"ada\": 1, \"bo\": 0} and len(g.messages.sent) == 1")

### Bug hunt: An adapter that forgot the units

The adapter fits the method name but not the units: it passes Fahrenheit through as Celsius, so the average is nonsense (`46.0`). Fix `FahrenheitAdapter.celsius` so this prints `21.0`.

In [ ]:
class OldSensor:
    def __init__(self, f):
        self.f = f
    def fahrenheit(self):
        return self.f

class NewSensor:
    def __init__(self, c):
        self.c = c
    def celsius(self):
        return self.c

class FahrenheitAdapter:
    def __init__(self, sensor):
        self.sensor = sensor
    def celsius(self):
        return self.sensor.fahrenheit()

sensors = [NewSensor(20.0), FahrenheitAdapter(OldSensor(71.6))]
print(round(sum(s.celsius() for s in sensors) / len(sensors), 1))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"converted\"\n    expect_output(\"21.0\")")

### Bug hunt: Filled in the wrong place

`partial` filled in the wrong argument: the text goes to a channel called `Ada`, addressed to `sms`. Fix the `partial` so this prints `[sms] to Ada: hello`.

In [ ]:
from functools import partial

def send(channel, to, text):
    return f"[{channel}] to {to}: {text}"

to_ada = partial(send, "Ada", "sms")
print(to_ada("hello"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the right arguments\"\n    expect_output(\"[sms] to Ada: hello\")")

### Bug hunt: Adapted by inheriting

`ListSource` was made to fit by inheriting from `list` and adding `read`, but `count_words` also calls `close()` on what it reads (as it would for a file), and lists have no `close`. Rewrite `ListSource` as an object adapter that holds the lines and offers exactly `read()` and `close()` (which does nothing), so this prints `5`.

In [ ]:
def count_words(source):
    try:
        return len(source.read().split())
    finally:
        source.close()

class ListSource(list):
    def read(self):
        return "\n".join(self)

print(count_words(ListSource(["three words here", "and two"])))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"an object adapter\"\n    expect_output(\"5\")\n    assert not issubclass(ns[\"ListSource\"], list), \"hold the lines instead of inheriting from list\"")

### Puzzle: An adapter for a printer

Arrange the lines so that an old printer with a `print_text` method fits code that calls `write`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from functools import partial

def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
two_to = partial(power, 2)

class Doubler:
    def __init__(self, f):
        self.f = f
    def __call__(self, x):
        return 2 * self.f(x)

p = ___   # square(5)
q = ___   # two_to(10)
r = ___   # Doubler(square)(3)
s = ___   # square.keywords
t = ___   # [f(3) for f in (square, two_to)]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (25, 1024, 18), \"p to r\"\n    assert s == {\"exp\": 2} and t == [9, 8]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: one calendar from three sources

Three calendar sources describe events in three different ways, and none of them can be changed. Adapt them all to one interface, `EventSource`, a Protocol with `events() -> list[Event]`, where `Event` is a frozen, ordered dataclass with `start` and `end` (`datetime`s) and `title`.

- `OldDiary` (given) has `entries()`, returning tuples like `("2026-11-03 09:00", 60, "Dentist")`: a start, a length in minutes, and a title. Write `DiaryAdapter`.
- `TeamPlanner` (given) has `fetch()`, returning dictionaries like `{"from": datetime(...), "to": datetime(...), "what": "Stand-up"}`. Write `PlannerAdapter`.
- `ShiftFile` (given) has `text`, lines like `EVENT|20261103T0900|20261103T1000|Training`. Write `ShiftAdapter`, ignoring blank lines.

Then write `merged(sources)`, every event from every source in time order (by start, then end, then title), and `clashes(events)`, the pairs of titles of events that overlap (one starts before the other ends; touching end to start isn't a clash), each pair as a tuple in time order, in order of the first event.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Protocol

class OldDiary:
    def __init__(self, rows):
        self._rows = rows
    def entries(self):
        return list(self._rows)

class TeamPlanner:
    def __init__(self, items):
        self._items = items
    def fetch(self):
        return [dict(i) for i in self._items]

class ShiftFile:
    def __init__(self, text):
        self.text = text

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def sources():\n    from datetime import datetime as dt\n    d = ns[\"OldDiary\"]([(\"2026-11-03 09:00\", 60, \"Dentist\"), (\"2026-11-03 13:00\", 45, \"Lunch with Kemi\")])\n    p = ns[\"TeamPlanner\"]([{\"from\": dt(2026, 11, 3, 9, 30), \"to\": dt(2026, 11, 3, 9, 45), \"what\": \"Stand-up\"}])\n    s = ns[\"ShiftFile\"](\"EVENT|20261103T1000|20261103T1200|Training\\n\\nEVENT|20261103T1330|20261103T1400|Handover\\n\")\n    return ns[\"DiaryAdapter\"](d), ns[\"PlannerAdapter\"](p), ns[\"ShiftAdapter\"](s)\ndef test_adapters():\n    \"the adapters\"\n    from datetime import datetime as dt\n    E = ns[\"Event\"]\n    d, p, s = sources()\n    assert d.events()[0] == E(dt(2026, 11, 3, 9, 0), dt(2026, 11, 3, 10, 0), \"Dentist\")\n    assert p.events() == [E(dt(2026, 11, 3, 9, 30), dt(2026, 11, 3, 9, 45), \"Stand-up\")]\n    assert s.events()[1] == E(dt(2026, 11, 3, 13, 30), dt(2026, 11, 3, 14, 0), \"Handover\") and len(s.events()) == 2\ndef test_merged():\n    \"merging and clashes\"\n    events = merged(sources())\n    assert [e.title for e in events] == [\"Dentist\", \"Stand-up\", \"Training\", \"Lunch with Kemi\", \"Handover\"]\n    assert clashes(events) == [(\"Dentist\", \"Stand-up\"), (\"Lunch with Kemi\", \"Handover\")], f\"got {clashes(events)}\"\n    assert merged([]) == [] and clashes([]) == []")

## Warm-up

`count_words` wants something with a `read` method. A list of lines doesn't have one, but a few lines of `LinesSource` give it one, translating "read everything" into "join the lines". Neither `count_words` nor the list had to change. That small translator is an **adapter**.

## Interfaces that don't match

Mismatches are everywhere in real programs:

- a library you didn't write has the right ability behind the wrong method names;
- old code works in different units, or returns tuples where new code wants objects;
- a callback is called with arguments your function doesn't take;
- data arrives from another program in a shape that doesn't match your classes.

Changing either side is often impossible (it's a library) or unwise (other code depends on it). Adding `if isinstance(x, OldThing)` everywhere couples your code to every mismatch (Week 34). An adapter keeps the translation in one small place.

## The Adapter pattern

The Gang of Four describe Adapter as: "convert the interface of a class into another interface clients expect. Adapter lets classes work together that couldn't otherwise because of incompatible interfaces." The usual Python form holds the adapted object and translates each call (an **object adapter**):

- `average` only knows the `Sensor` interface, and works with both kinds.
- The adapter does the **translation** (method name and units) and nothing more. It doesn't add features: that would be a different job.
- Holding the object (composition) means one adapter class works with any `OldThermometer`, and the thermometer's own methods aren't mixed into the adapter.

An adapter could also **inherit** from the adapted class (a "class adapter"), but then it carries every method of the old class, including ones that don't fit the new interface, and it can only adapt that one class. Week 30, Day 5's advice holds: prefer holding to inheriting.

## Adapting functions

Callbacks often need small adapters. Suppose a notifier calls each handler with one argument, the message, but the function you want to use needs more:

`functools.partial(func, *args, **kwargs)` makes a new callable that calls `func` with those arguments already filled in, plus whatever it's called with. Positional arguments given to `partial` fill the **first** parameters; anything the caller then passes by position goes into the next ones. So a parameter filled in by keyword must come **after** every parameter the caller will pass by position; otherwise the caller must pass the others by keyword (as in the next example). It's an adapter for signatures: here, three-argument `send` becomes three different one-argument handlers. A `lambda` does the same and can also reorder or convert arguments; `partial` is clearer when you're only filling some in, and its objects show what they hold:

Adapting **results** works the same way: a small function that calls the original and reshapes what it returns.

## Adapting data

Data from outside your program is the commonest mismatch of all. Two weather services, say, send the same information in different shapes, and your program wants its own dataclass. A **data adapter** is a function per source that converts one shape into yours, at the boundary (Week 33, Day 5):

The rest of the program only ever sees `Forecast`. If a service changes its format, or a third is added, only one small function changes or appears.

## Facade

The **Facade** pattern (also from the Gang of Four) is a close cousin: instead of making one thing fit an interface, it puts **one simple interface in front of several parts**. Buying something online involves stock, payments and email; most callers just want "buy this":

The facade knows the order of the steps and how to undo them if payment fails; callers just call `buy`. The parts are still there for code that needs them directly. Adapter and Facade both wrap, but for different reasons: an adapter **changes** an interface to fit, a facade **simplifies** many into one.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a library or service you've used whose data or method names didn't match what your code wanted, and sketch the adapter you'd write. Tomorrow: the Decorator pattern, which adds behaviour by wrapping an object or function in another with the same interface.